# Tarea Hogar 04: ensamble robusto (z499)

Experimento **HT4990**. Cierra z495–z498 con un diagnóstico completo y un modelo final pensado para el **Private**, no para el Public.

## 1. Qué pasó hasta acá

| subida | qué es | Public |
|---|---|---|
| KA5940 | cátedra (z494): BAJA+1+2, 100 hojas, lr 0.05, 500 árboles, max_bin 31 | 353–376 según cupo (369.4 a 11000) |
| KA495 | HPO, solo BAJA+2 | 345–360 |
| KA497 | HPO en dos etapas, solo BAJA+2, 15 hojas | 348–350 |
| KA498 V1 | centro de z497 (8 hojas) + BAJA+1+2 | 363.6 |
| KA498 V2/V3 | V1 sin fechas / rank por mes | 362.3 / 362.7 |
| KA498 V4 | V3 + variables nuevas (cocientes) | 343.6 |

## 2. Cuánto ruido tiene el Public

El Public es un **30% al azar de 202109**. El puntaje es la ganancia de ese 30% dividida por 0.3, en millones. Se puede despejar de los 7 cupos de KA5940: cada 500 envíos hay ~150 clientes en el Public, y cada BAJA+2 que entra suma **3.25 puntos**.

- De 10000 a 10500 envíos KA5940 subió 10.9 puntos. Eso son **7 BAJA+2** del Public. De 10500 a 11000 bajó 6.5: entraron **2**. El "mejor cupo 10500" es ruido de conteo.
- Simulando particiones 30/70 sobre predicciones fuera de fold de 202107, se obtiene lo siguiente:
  - el puntaje de un modelo tiene un desvío de **±43 puntos**;
  - la diferencia entre dos modelos cuyos top-11000 comparten el 80% tiene un desvío de **±17 puntos**;
  - si simulás qué cupo gana en el Public, sale cualquiera entre 9000 y 13000.
- El 1ro del leaderboard tiene 393 y vos 375.9: **17 puntos, unos 5 BAJA+2**. No es una diferencia que el Public pueda resolver.

**Conclusión: el Public no sirve para elegir entre modelos parecidos.** Sirve para detectar un desastre de más de 30 o 40 puntos. El Private (el otro 70%) es el que cuenta.

## 3. Qué dice la validación local (5-fold × 3 repeticiones, fuera de fold, mes entero)

Todo pareado con la misma partición. "Meseta" es la ganancia media entre 9000 y 13000 envíos, en millones sobre 164k clientes.

| modelo | meseta | vs KA5940 |
|---|---|---|
| KA5940, solo BAJA+2 | 522 | **−43 ± 2** |
| KA5940 (receta cátedra) | 565 | 0 |
| KA5940 sin `numero_de_cliente` | 566 | +0.4 ± 3.9 |
| KA5940 con max_bin 255 | 567 | +1.7 ± 6.7 |
| KA5940 con ff 0.5 y bagging 0.8 | 561 | −4.3 ± 3.3 |
| 128 hojas regularizado (min_data 200, λ2 10, ff .5) | 569 | +3.3 ± 4.5 |
| 32 hojas regularizado | 576 | +10.9 ± 3.2 |
| 4 hojas | 570 | +5.0 ± 3.8 |
| 8 hojas (centro z497) | 582 | **+17.2 ± 3.7** |
| 8 hojas, lr 0.01, 2700 árboles | 582 | +17.1 ± 3.6 |
| 16 hojas, λ2 5, ff .5 | 583 | +17.5 ± 6.0 |
| 8 hojas, BAJA+1 con peso 0.5 | 583 | +17.2 ± 1.3 |
| 8 hojas sin ninguna fecha | 581 | +15.4 ± 4.3 |
| **ensamble KA5940 + 8 hojas lento (rank medio)** | **584** | **+18.5 ± 2.2** |

Lo que queda firme:

1. **Label BAJA+1+2: +43M.** Es lo único grande, y ya lo sabíamos (z494 lo tenía, z495–z497 no).
2. **Árboles chicos (8–16 hojas) > árboles de 100 hojas, +17M en 202107.** Pero en 202109 KA5940 sacó 369.4 y V1 363.6 al mismo cupo. La diferencia, −6 ± 17, no alcanza para decir que se invierte. Aun así es una alerta: la validación dentro de julio puede premiar cosas que no viajan a septiembre.
3. Lo demás son empates: max_bin, `numero_de_cliente`, fechas, peso de BAJA+1, lr más chico.
4. **Las variables nuevas (V4) bajaron 20 puntos en el Public con +3.6 ± 3.4 local.** No es concluyente (±17), pero no hay evidencia a favor. Quedan afuera.

## 4. Qué dice la documentación de LightGBM

[Parameters-Tuning](https://lightgbm.readthedocs.io/en/latest/Parameters-Tuning.html) separa dos listas:

- **Para más precisión:** `max_bin` grande, `learning_rate` chico con muchos `num_iterations`, `num_leaves` grande, más datos, `dart`.
- **Contra el overfitting:** `max_bin` chico, `num_leaves` chico, `min_data_in_leaf` y `min_sum_hessian_in_leaf`, bagging (`bagging_fraction` + `bagging_freq`), `feature_fraction`, `lambda_l1`/`lambda_l2`/`min_gain_to_split`, `max_depth`, `extra_trees`, `path_smooth`.

Aplicado a este problema:

- Hay 164k filas con solo 1304 BAJA+2, y cambia el mes. La lista contra el overfitting es la que importa, y los datos lo confirman: 8 hojas le gana a 100.
- **`min_data_in_leaf` depende de la cantidad de filas** ("its optimal value depends on the number of training samples and num_leaves"). Por eso el modelo final, que ve el 100% del mes, lo reescala desde el fold, que ve el 80%.
- **`seed` solo cambia algo si hay azar.** Con `feature_fraction = 1` y sin bagging, KA5940 es determinístico: un semillerío sobre esa receta promedia 20 copias del mismo modelo. El semillerío se aplica al modelo chico, que usa `feature_fraction = 0.8`.
- **`feature_pre_filter = FALSE`** es obligatorio si `min_data_in_leaf` cambia entre el Dataset y el entrenamiento (fold → final). Si no, LightGBM descarta variables con el valor viejo.
- `is_unbalance`/`scale_pos_weight` cambian la calibración pero no el orden. Como se envía por ranking (top-k), no aportan.

## 5. La decisión

No se puede validar el cambio de mes (hay un solo mes con clase). Entonces se usa la opción que **no depende de adivinar cuál validación tiene razón**:

- **Modelo A: la receta KA5940 tal cual.** Es la única con evidencia buena en 202109.
- **Modelo B: 8 hojas, lr 0.01, 2700 árboles, semillerío de 20.** Es la mejor en 202107, y el lr lento es lo que recomienda la documentación.
- **Final: promedio de los percentiles de A y B, 50/50.** El peso **se fija de antemano**: si se optimizara sobre julio, se volvería a sesgar hacia B. Localmente el ensamble no pierde nada contra el mejor (584 contra 582) y le gana 18M a KA5940. En septiembre queda cubierto contra el caso en que A generalice mejor.
- **Cupo 11000.** Es el centro de la meseta local, que es plana entre 10500 y 12500. El 10500 del Public no es evidencia.
- **Una sola subida al Public.** Sirve de chequeo de desastre, no para elegir. Para el Private conviene marcar el ensamble a 11000 y, como segunda, KA5940 a 11000, no a 10500.

In [ ]:
if (!require("data.table")) install.packages("data.table")
if (!require("lightgbm")) install.packages("lightgbm")
require("data.table")
require("lightgbm")
require("parallel")

setDTthreads(percent = 100)
options(scipen = 999)

## 1. PARAM

Dos modelos fijos: no se tunea nada acá. `min_data_in_leaf` está expresado **para el 100% de 202107**. La validación lo reescala al tamaño del fold (80%).

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 271211L
PARAM$estudiante <- "Maceo, Marcos"
PARAM$experimento <- "HT4990"
PARAM$mc_cores <- max(1L, detectCores() - 1L)

# +++ validacion: 5-fold estratificado, repetido; ganancia sobre las predicciones fuera de fold del mes entero
PARAM$folds <- 5L
PARAM$repeticiones <- c(271211L, 200177L, 410551L)
PARAM$meseta <- c(9000L, 13000L)     # ganancia media en este rango de envios (mas estable que el maximo)

FECHAS_CIERRE <- c("Visa_fultimo_cierre", "Master_fultimo_cierre")   # constantes dentro del mes: solo marcan el mes
BASE_LGB <- list(
  objective = "binary", metric = "auc", boosting = "gbdt", verbosity = -1,
  feature_pre_filter = FALSE, force_row_wise = TRUE, boost_from_average = TRUE,
  max_depth = -1L, min_gain_to_split = 0, min_sum_hessian_in_leaf = 0.001,
  lambda_l1 = 0, lambda_l2 = 0, bagging_fraction = 1, bagging_freq = 0L, feature_fraction = 1
)

PARAM$modelos <- list(
  # +++ A: la receta de KA5940 (z494) tal cual. Deterministica (ff = 1, sin bagging): una sola semilla alcanza
  A = list(lgb = list(num_leaves = 100L, learning_rate = 0.05, min_data_in_leaf = 40L),
           num_iterations = 500L, max_bin = 31L, sacar = character(), semillas = 1L),
  # +++ B: 8 hojas, lr lento. ff 0.8 da azar entre semillas: semillerio de 20
  B = list(lgb = list(num_leaves = 8L, learning_rate = 0.01, min_data_in_leaf = 109L, feature_fraction = 0.8),
           num_iterations = 2700L, max_bin = 127L, sacar = c("numero_de_cliente", FECHAS_CIERRE), semillas = 20L)
)
PARAM$peso_A <- 0.5                   # +++ fijado de antemano, no se optimiza sobre 202107

PARAM$cupos_csv <- seq(9000L, 13000L, by = 500L)
PARAM$cupo_kaggle <- 11000L           # +++ una sola subida: chequeo de desastre, no seleccion
PARAM$mc_cores

## 2. Dataset

In [ ]:
candidatos_exp <- c("/content/buckets/b1/exp", path.expand("~/buckets/b1/exp"), file.path(getwd(), "exp"))
base_exp <- candidatos_exp[dir.exists(candidatos_exp)][1]
if (is.na(base_exp)) {
  base_exp <- candidatos_exp[3]
  dir.create(base_exp, recursive = TRUE, showWarnings = FALSE)
}
dir.create(file.path(base_exp, PARAM$experimento), showWarnings = FALSE)
setwd(file.path(base_exp, PARAM$experimento))
getwd()

candidatos_ds <- c(
  "/content/datasets/dataset_pequeno.csv",
  path.expand("~/datasets/dataset_pequeno.csv"),
  path.expand("~/buckets/b1/datasets/dataset_pequeno.csv")
)
archivo_dataset <- candidatos_ds[file.exists(candidatos_ds)][1]
stopifnot(!is.na(archivo_dataset))

dataset <- fread(archivo_dataset)
dataset[, intersect(c("clase01", "azar", "training"), names(dataset)) := NULL]
dataset_mes <- dataset[foto_mes == 202107]
dataset[, .N, .(foto_mes, clase_ternaria)]

## 3. Validación: 5-fold × 3 repeticiones, fuera de fold

Cada cliente de 202107 recibe una predicción de un modelo que no lo vio. La curva de ganancia se arma sobre **el mes entero** (164k clientes), no sobre un 30%. Eso tiene menos ruido que un holdout y es directamente comparable con la escala de 202109.

Los dos modelos usan los mismos folds en cada repetición, así que la comparación es pareada. El ensamble promedia los percentiles fuera de fold de A y B. Se hace por repetición, con los mismos folds.

Se puede cortar y retomar: cada (modelo, repetición) se graba apenas termina.

In [ ]:
dataset_mes[, clase01 := as.integer(clase_ternaria %in% c("BAJA+1", "BAJA+2"))]
campos_de <- function(m) setdiff(names(dataset_mes), c("clase_ternaria", "clase01", "foto_mes", m$sacar))
X_mes <- data.matrix(dataset_mes[, setdiff(names(dataset_mes), c("clase_ternaria", "clase01", "foto_mes")), with = FALSE])
gan_mes <- ifelse(dataset_mes$clase_ternaria == "BAJA+2", 975000, -25000)

folds_de <- function(semilla) {
  set.seed(semilla)
  f <- integer(nrow(dataset_mes))
  for (k in unique(dataset_mes$clase_ternaria)) {
    i <- which(dataset_mes$clase_ternaria == k)
    f[i] <- sample(rep_len(seq_len(PARAM$folds), length(i)))
  }
  f
}

fit_fold <- function(job) {
  m <- PARAM$modelos[[job$modelo]]
  f <- folds_de(job$rep)
  tr <- f != job$fold
  cols <- campos_de(m)
  # +++ min_data_in_leaf pensado para el 100%: el fold ve (k-1)/k
  frac <- (PARAM$folds - 1) / PARAM$folds
  p <- modifyList(modifyList(BASE_LGB, m$lgb),
                  list(num_threads = 1L, seed = job$rep,
                       min_data_in_leaf = as.integer(round(m$lgb$min_data_in_leaf * frac))))
  ds <- lightgbm::lgb.Dataset(X_mes[tr, cols], label = dataset_mes$clase01[tr],
                              params = list(max_bin = m$max_bin, feature_pre_filter = FALSE))
  mod <- lightgbm::lgb.train(p, ds, nrounds = m$num_iterations, verbose = -1)
  data.table::data.table(modelo = job$modelo, rep = job$rep, idx = which(!tr),
                         prob = predict(mod, X_mes[!tr, cols]))
}

dir.create("oof", showWarnings = FALSE)
archivo_oof <- function(modelo, rep) sprintf("oof/oof_%s_%d.rds", modelo, rep)
pend <- list()
for (mo in names(PARAM$modelos)) for (r in PARAM$repeticiones) {
  if (!file.exists(archivo_oof(mo, r))) pend[[length(pend) + 1]] <- list(modelo = mo, rep = r)
}
cat("pendientes (modelo x repeticion):", length(pend), "\n")

if (length(pend)) {
  cl <- makeCluster(PARAM$mc_cores, type = "PSOCK")
  clusterEvalQ(cl, {
    Sys.setenv(OMP_NUM_THREADS = "1")
    suppressPackageStartupMessages({ library(data.table); library(lightgbm) })
    data.table::setDTthreads(1)
    NULL
  })
  clusterExport(cl, c("fit_fold", "folds_de", "campos_de", "X_mes", "dataset_mes", "PARAM", "BASE_LGB"), envir = .GlobalEnv)
  t0 <- Sys.time()
  for (pj in pend) {
    jobs <- lapply(seq_len(PARAM$folds), function(k) c(pj, fold = k))
    oof <- rbindlist(parLapplyLB(cl, jobs, fit_fold))
    saveRDS(oof, archivo_oof(pj$modelo, pj$rep))
    cat(sprintf("%s rep %d listo | %.1f min\n", pj$modelo, pj$rep, as.numeric(difftime(Sys.time(), t0, units = "mins"))))
    flush.console()
  }
  stopCluster(cl)
}

In [ ]:
oof <- rbindlist(lapply(list.files("oof", full.names = TRUE), readRDS))
oof[, pct := frank(prob) / .N, by = .(modelo, rep)]
ens <- dcast(oof, rep + idx ~ modelo, value.var = "pct")
ens <- ens[, .(modelo = "ENS", rep, idx, pct = PARAM$peso_A * A + (1 - PARAM$peso_A) * B)]
todo_oof <- rbind(oof[, .(modelo, rep, idx, pct)], ens)

curva <- function(pct, idx) cumsum(gan_mes[idx][order(-pct)])
met <- todo_oof[, {
  cs <- curva(pct, idx)
  s <- frollmean(cs, 401L, align = "center")
  .(meseta = mean(cs[PARAM$meseta[1]:PARAM$meseta[2]]) / 1e6,
    g_cupo = cs[PARAM$cupo_kaggle] / 1e6,
    max_suave = max(s, na.rm = TRUE) / 1e6, k_max = which.max(s))
}, by = .(modelo, rep)]

w <- dcast(met, rep ~ modelo, value.var = "meseta")
resumen <- met[, .(meseta_M = round(mean(meseta), 1), ganancia_cupo_M = round(mean(g_cupo), 1),
                   max_suave_M = round(mean(max_suave), 1), k_max = as.integer(median(k_max))), by = modelo]
resumen[, delta_vs_A := round(sapply(modelo, function(m) mean(w[[m]] - w$A)), 1)]
resumen[, se_delta := round(sapply(modelo, function(m) sd(w[[m]] - w$A) / sqrt(nrow(w))), 1)]
fwrite(resumen, "resumen_th04_z499.tsv", sep = "\t")
print(resumen)

# +++ curva media del ensamble: donde esta la meseta
cur <- todo_oof[modelo == "ENS", .(k = seq_len(.N), g = curva(pct, idx)), by = rep][, .(g = mean(g) / 1e6), by = k]
print(cur[k %in% PARAM$cupos_csv])
cat(sprintf("\nSolapamiento top-%d entre A y B (fuera de fold, rep 1): %.3f\n", PARAM$cupo_kaggle,
    oof[rep == PARAM$repeticiones[1], .(top = list(idx[order(-prob)][seq_len(PARAM$cupo_kaggle)])), by = modelo][
      , length(intersect(top[[1]], top[[2]])) / PARAM$cupo_kaggle]))

## 4. Modelo final, CSV y Kaggle

- Se entrena sobre **todo** 202107, sin undersampling, con los `min_data_in_leaf` de PARAM, que ya están en escala del 100%.
- A es un solo modelo. B promedia las probabilidades de 20 semillas.
- El score final de cada cliente de 202109 es `0.5 · percentil(A) + 0.5 · percentil(B)`.
- Se graban los CSV de 9000 a 13000 envíos.
- A Kaggle se sube **una sola** subida, a `PARAM$cupo_kaggle`. Si baja más de ~35 puntos respecto de KA5940 a 11000 (369.4), algo se rompió. Cualquier otra diferencia es ruido.

In [ ]:
CORRER_KAGGLE <- FALSE   # +++ TRUE para subir el ensamble a PARAM$cupo_kaggle (una sola subida)

dfut <- dataset[foto_mes == 202109]
X_fut <- data.matrix(dfut[, setdiff(names(dfut), c("clase_ternaria", "clase01", "foto_mes")), with = FALSE])
X_tr <- X_mes

probs <- list()
for (mo in names(PARAM$modelos)) {
  archivo <- sprintf("KA499_%s_prob.csv", mo)
  if (file.exists(archivo)) {
    probs[[mo]] <- fread(archivo)$prob
    cat(mo, ": ya estaba, la reuso\n")
    next
  }
  m <- PARAM$modelos[[mo]]
  cols <- campos_de(m)
  ds <- lgb.Dataset(X_tr[, cols], label = dataset_mes$clase01,
                    params = list(max_bin = m$max_bin, feature_pre_filter = FALSE), free_raw_data = FALSE)
  set.seed(PARAM$semilla_primigenia)
  semillas <- sample(100000:999999, m$semillas)
  p <- numeric(nrow(X_fut))
  t0 <- Sys.time()
  for (s in semillas) {
    mod <- lgb.train(modifyList(modifyList(BASE_LGB, m$lgb), list(num_threads = PARAM$mc_cores + 1L, seed = s)),
                     ds, nrounds = m$num_iterations, verbose = -1)
    p <- p + predict(mod, X_fut[, cols]) / length(semillas)
  }
  if (mo == "A") fwrite(as.data.table(lgb.importance(mod)), "impo_A.tsv", sep = "\t")
  if (mo == "B") fwrite(as.data.table(lgb.importance(mod)), "impo_B.tsv", sep = "\t")
  probs[[mo]] <- p
  fwrite(data.table(numero_de_cliente = dfut$numero_de_cliente, prob = p), archivo)
  cat(sprintf("%s: %d semillas, %.1f min\n", mo, length(semillas), as.numeric(difftime(Sys.time(), t0, units = "mins"))))
  flush.console()
}

score <- PARAM$peso_A * frank(probs$A) / length(probs$A) + (1 - PARAM$peso_A) * frank(probs$B) / length(probs$B)
tb <- data.table(numero_de_cliente = dfut$numero_de_cliente, score = score)
fwrite(tb, "KA499_ENS_score.csv")
ord <- order(-tb$score)
for (envios in PARAM$cupos_csv) {
  pred <- integer(nrow(tb))
  pred[ord[seq_len(envios)]] <- 1L
  fwrite(data.table(numero_de_cliente = tb$numero_de_cliente, Predicted = pred), sprintf("KA499_%05d.csv", envios))
}
top <- function(p) order(-p)[seq_len(PARAM$cupo_kaggle)]
cat(sprintf("Solapamiento top-%d en 202109: A-B %.3f | A-ENS %.3f | B-ENS %.3f\n", PARAM$cupo_kaggle,
    length(intersect(top(probs$A), top(probs$B))) / PARAM$cupo_kaggle,
    length(intersect(top(probs$A), top(score))) / PARAM$cupo_kaggle,
    length(intersect(top(probs$B), top(score))) / PARAM$cupo_kaggle))

if (CORRER_KAGGLE) {
  archivo <- sprintf("KA499_%05d.csv", PARAM$cupo_kaggle)
  linea <- sprintf("kaggle competitions submit -c labo-1-ba-inicial -f %s -m 'z499 ENS A(KA5940)+B(8h lr.01 x20) 50/50 envios=%d'",
                   archivo, PARAM$cupo_kaggle)
  cat(system(linea, intern = TRUE), sep = "\n")
}

## 5. Planilla

Una fila para la hoja **TareaHogar-04**. La ganancia local es la meseta fuera de fold del ensamble, en la escala del mes entero.

In [ ]:
r <- fread("resumen_th04_z499.tsv")
planilla <- data.table(
  estudiante = PARAM$estudiante, experimento = PARAM$experimento,
  modelo = "ENS 50/50: A=KA5940 (100h lr.05 500it mb31) + B=8h lr.01 2700it ff.8 x20 semillas",
  label = "BAJA+1 y BAJA+2", validacion = sprintf("5-fold x %d rep, OOF mes entero", length(PARAM$repeticiones)),
  ganancia_local_M = r[modelo == "ENS", meseta_M], delta_vs_KA5940_M = r[modelo == "ENS", delta_vs_A],
  envios = PARAM$cupo_kaggle, kaggle_public = NA_real_
)
fwrite(planilla, "planilla_TH04_z499.tsv", sep = "\t")
planilla